# Road Following – Calibrated Live Demo

This notebook is identical in structure to `live_demo_executelock.ipynb` but seeds
the steering sliders from **`steering_calibration.json`** produced by
`steering_calibration.ipynb`.

The calibrated `steering_gain` and `base_speed` are loaded automatically.
All sliders remain editable – the calibration values are simply used as the
starting point instead of hard-coded defaults.

> Make sure `steering_calibration.json` is in the same directory as this notebook
> before running Cell 1.

## 1. Load Calibration

In [ ]:
import json

CALIB_PATH = 'steering_calibration.json'

with open(CALIB_PATH) as f:
    calib = json.load(f)

CALIB_STEERING_GAIN = calib['steering_gain']   # motor-diff per degree of yaw
CALIB_BASE_SPEED    = calib['base_speed']
CALIB_CORRECTION    = calib.get('correction_gain', 0.0)

print(f'Loaded calibration v{calib["version"]}  ({calib["trials"]} trials, '
      f'converge rate {calib["converge_rate"]*100:.0f}%)')
print(f'  steering_gain  = {CALIB_STEERING_GAIN}')
print(f'  base_speed     = {CALIB_BASE_SPEED}')
print(f'  correction_gain= {CALIB_CORRECTION}')

## 2. Load Trained Model

In [ ]:
MODEL_PATH = 'models/dataset_labeled_2/model_to_import.pth'

In [ ]:
import torchvision
import torch

model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)
model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
device = torch.device('cuda')
model  = model.to(device)
model  = model.eval().half()

dummy = torch.zeros(1, 3, 224, 224).to(device).half()
with torch.no_grad():
    model = torch.jit.trace(model, dummy)
    for _ in range(20):
        model(dummy)

print('Model ready')

## 3. Pre-Processing

In [ ]:
import torchvision.transforms as transforms
import cv2
import PIL.Image
import numpy as np

mean_gpu = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_gpu  = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_gpu(image):
    """Convert camera BGR numpy array to normalised GPU half-tensor."""
    tensor = torch.from_numpy(image).permute(2, 0, 1).float()  # HWC -> CHW
    tensor = tensor[[2, 1, 0]]           # BGR -> RGB
    tensor = tensor.to(device).half()
    tensor.div_(255.0)
    tensor.sub_(mean_gpu[:, None, None]).div_(std_gpu[:, None, None])
    return tensor.unsqueeze(0)

def predict_xy(image):
    tensor = preprocess_gpu(image)
    with torch.no_grad():
        xy = model(tensor).squeeze()
    return float(xy[0]), float(xy[1])

## 4. Camera & Robot

In [ ]:
from IPython.display import display
import ipywidgets
import traitlets
from jetbot import Camera, bgr8_to_jpeg

camera = Camera()

image_widget = ipywidgets.Image()
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

In [ ]:
from jetbot import Robot

robot = Robot()
print('Robot ready.')

## 5. Control Sliders

Slider **initial values are seeded from the calibration file**.
Adjust them live to fine-tune behaviour on the track.

| Slider | Calibration source | Purpose |
|---|---|---|
| `speed_gain` | `calib['base_speed']` | Overall drive speed |
| `steering_gain` | `calib['steering_gain']` | P-term – how hard to steer per unit angle |
| `steering_kd` | 0 (tune manually) | D-term – dampens oscillation |
| `steering_bias` | 0 (tune manually) | Corrects persistent left/right drift |

In [ ]:
speed_gain_slider    = ipywidgets.FloatSlider(
    min=0.0, max=1.0, step=0.01,
    value=CALIB_BASE_SPEED,
    description='speed gain')

steering_gain_slider = ipywidgets.FloatSlider(
    min=0.0, max=1.0, step=0.001,
    value=min(CALIB_STEERING_GAIN, 1.0),   # clamp – calibration gain is in deg units
    description='steering gain',
    readout_format='.3f')

steering_dgain_slider = ipywidgets.FloatSlider(
    min=0.0, max=0.5, step=0.001,
    value=0.0,
    description='steering kd',
    readout_format='.3f')

steering_bias_slider = ipywidgets.FloatSlider(
    min=-0.3, max=0.3, step=0.01,
    value=0.0,
    description='steering bias')

# Info label showing calibration provenance
calib_label = ipywidgets.HTML(
    value=f'<b>Calibration loaded:</b> {calib["trials"]} trials &nbsp;|&nbsp; '
          f'converge {calib["converge_rate"]*100:.0f}% &nbsp;|&nbsp; '
          f'avg final yaw {calib["avg_final_yaw_deg"]:.2f}&deg;'
)

display(calib_label,
        speed_gain_slider, steering_gain_slider,
        steering_dgain_slider, steering_bias_slider)

## 6. Diagnostic Sliders

In [ ]:
x_slider        = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='x')
y_slider        = ipywidgets.FloatSlider(min=0,    max=1.0, orientation='vertical', description='y')
steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='steering')
speed_slider    = ipywidgets.FloatSlider(min=0,    max=1.0, orientation='vertical', description='speed')

display(ipywidgets.HBox([y_slider, speed_slider]))
display(x_slider, steering_slider)

## 7. Execute Function (with execute-lock)

Identical to `live_demo_executelock.ipynb` – the only difference is that
`steering_gain_slider` and `speed_gain_slider` start from calibration values.

In [ ]:
import threading
from time import time

execute_times = []
executing     = False
execute_lock  = threading.Lock()

angle      = 0.0
angle_last = 0.0

def execute(change):
    now = time()
    execute_times.append(now)
    execute_times[:] = [t for t in execute_times if t > now - 2]

    global angle, angle_last, executing

    with execute_lock:
        if executing:
            return
        executing = True

    try:
        image = change['new']
        pred_x, pred_y = predict_xy(image)

        x = pred_x - 0.5
        y = 1.0 - pred_y
        angle = np.arctan2(x, y)

        pid = (angle * steering_gain_slider.value
               + (angle - angle_last) * steering_dgain_slider.value)
        angle_last = angle

        steering = pid + steering_bias_slider.value

        x_slider.value        = x
        y_slider.value        = y
        speed_slider.value    = speed_gain_slider.value
        steering_slider.value = steering

        left  = float(np.clip(speed_gain_slider.value + steering, 0.0, 1.0))
        right = float(np.clip(speed_gain_slider.value - steering, 0.0, 1.0))

        robot.left_motor.value  = left
        robot.right_motor.value = right

    finally:
        with execute_lock:
            executing = False

# Warm-up call
execute({'new': camera.value})
print('execute() ready.')

## 8. Start Road Following

> **WARNING: The robot will move!**  
> Make sure it has clearance and is on the track used for data collection.

In [ ]:
camera.observe(execute, names='value')

## 9. Stop Road Following

In [ ]:
import time as _time

camera.unobserve(execute, names='value')
_time.sleep(0.1)
robot.stop()

## 10. Release Camera

In [ ]:
camera.stop()